# USDA RMA Cause of Loss — *Why* Crops Failed (Insurance Records)

**What it is.** The Risk Management Agency's crop-insurance loss records, summarized by
**cause of loss** (drought, hail, excess moisture, freeze, heat, wind…). For every
county × crop × year you get insured acres, liability, premium, and **indemnities paid**,
broken out by what actually caused the damage — and the **month of loss**.

| | |
|---|---|
| **Coverage** | United States — county × crop × cause × month |
| **History** | 1989 → present |
| **Cost / key** | **Free · no key** (bulk pipe-delimited files) |
| **Files** | `.../cause_of_loss/colsom_<YEAR>.zip` |
| **Docs** | https://www.rma.usda.gov/tools-reports/summary-of-business/cause-loss |

**Why it's UNIQUE to Tracera.** Almost no farm app surfaces this. It answers *"what actually
takes crops down in my county, how often, and how bad?"* — a data-driven risk profile you
can't get from weather alone.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Download & parse a year of cause-of-loss records (headerless, 30 fields)

In [2]:
import io, zipfile, pathlib

RMA_BASE = "https://pubfs-rma.fpac.usda.gov/pub/Web_Data_Files/Summary_of_Business/cause_of_loss"
# Column layout per the RMA "Summary of Business with Month of Loss" record layout PDF:
RMA_COLS = ["commodity_year","state_code","state_abbrev","county_code","county_name",
    "commodity_code","commodity_name","insurance_plan_code","insurance_plan_abbrev",
    "coverage_category","stage_code","cause_code","cause_of_loss","month_code","month_abbrev",
    "year_of_loss","policies_earning_prem","policies_indemnified","net_planted_qty",
    "net_endorsed_acres","liability","total_premium","producer_paid_premium","subsidy",
    "state_private_subsidy","additional_subsidy","efa_premium_discount","net_determined_qty",
    "indemnity_amount","loss_ratio"]

def rma_cause_of_loss(year, cache=pathlib.Path("data_cache")):
    cache.mkdir(exist_ok=True)
    fzip = cache / f"colsom_{year}.zip"
    if not fzip.exists():
        fzip.write_bytes(requests.get(f"{RMA_BASE}/colsom_{year}.zip", timeout=180).content)
    with zipfile.ZipFile(fzip) as z:
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f, sep="|", header=None, names=RMA_COLS,
                             encoding="latin-1", dtype=str)
    for c in ("liability","total_premium","indemnity_amount","net_planted_qty",
              "net_determined_qty","loss_ratio"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("state_abbrev","county_name","commodity_name","cause_of_loss"):
        df[c] = df[c].str.strip()
    return df

col = rma_cause_of_loss(2023)
print(f"OK — {len(col):,} loss records nationwide in 2023")
col.head(3)

OK — 147,309 loss records nationwide in 2023


,commodity_year,state_code,state_abbrev,county_code,county_name,commodity_code,commodity_name,insurance_plan_code,insurance_plan_abbrev,coverage_category,...,liability,total_premium,producer_paid_premium,subsidy,state_private_subsidy,additional_subsidy,efa_premium_discount,net_determined_qty,indemnity_amount,loss_ratio
0,2023,01,AL,001,Autauga,0021,Cotton,02,RP,A,...,78691.5,6922.0,3114.5000000000,3807.5000000000,.0000000000,.0000000000,.0000000000,98.30,2367.5,0.34
1,2023,01,AL,001,Autauga,0021,Cotton,02,RP,A,...,993832.5,84294.5,20620.5000000000,63674.0000000000,.0000000000,.0000000000,.0000000000,1575.54,234798.5,2.79
2,2023,01,AL,001,Autauga,0021,Cotton,02,RP,A,...,567298.0,49371.5,11065.0000000000,38306.5000000000,.0000000000,.0000000000,.0000000000,873.00,166030.0,3.36


### 2. Core analysis — top causes of loss & indemnities for our county

In [3]:
county = col[(col.state_abbrev == STATE) & (col.county_name.str.upper() == COUNTY)]
by_cause = (county.groupby("cause_of_loss")
                  .agg(indemnity_usd=("indemnity_amount", "sum"),
                       acres_lost=("net_determined_qty", "sum"))
                  .sort_values("indemnity_usd", ascending=False))
print(f"{COUNTY} County, {STATE} — 2023 crop-insurance losses by cause:")
by_cause.head(10)

STORY County, IA — 2023 crop-insurance losses by cause:


,indemnity_usd,acres_lost
cause_of_loss,,
Drought,638225.95,10190.1720
Decline in Price,485472.55,6217.3443
ARPI/SCO/ECO/STAX/MP/HIP-WI/PACE Crops Only,68032.00,0.0000
Hail,14053.40,172.9100
Heat,11413.50,227.0225
Cold Wet Weather,6944.60,168.2400
Excess Moisture/Precipitation/Rain,3923.00,66.9000
Wildlife,1382.00,7.7000


### Notes & how to extend
- **Multi-year risk profile:** loop `rma_cause_of_loss(y)` over 1989–present and rank causes
  by frequency & severity to build a county risk fingerprint.
- **Key fields:** `indemnity_amount` ($ paid), `net_determined_qty` (acres with a determined
  loss), `loss_ratio` (indemnity ÷ premium), `month_abbrev` (seasonality of losses).
- **Companion file:** the **Summary of Business** files (same host) add insured/liability totals
  without the cause breakout.
- Files are cached under `data_cache/` (git-ignored) so you download each year only once.